In [ ]:
%pip install -q wandb huggingface_hub datasets transformers tokenizers deepspeed

In [ ]:
import os
os.environ['HF_HOME'] = '/kaggle/working/hf_hub'

import json
import math
import shutil
import hashlib
import subprocess
import sys
from pathlib import Path

import numpy as np
import pyarrow as pa
from tqdm.auto import tqdm
import torch
from datasets import Dataset, load_dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from huggingface_hub import HfApi, snapshot_download
from kaggle_secrets import UserSecretsClient

assert torch.cuda.device_count() >= 2, 'Select GPU T4 x2 in Kaggle settings'
for index in range(2):
    print(index, torch.cuda.get_device_name(index))

workspace = Path('/kaggle/working/moe60m_ddp')
output_dir = workspace / 'output'
output_dir.mkdir(parents=True, exist_ok=True)
resume_training = False
experiment_name = 'prenorm_rmsnorm_swiglu_relative_moe_60m_ddp'
checkpoint_prefix = 'training/moe60m_10m_ddp'

secrets = UserSecretsClient()
hf_token = secrets.get_secret('huggingface-cli')
wandb_token = secrets.get_secret('wandb')
api = HfApi(token=hf_token)
repo_id = f"{api.whoami()['name']}/hw2-cg-{experiment_name}"
api.create_repo(repo_id=repo_id, repo_type='model', private=True, exist_ok=True)

config = {
    'seed': 67, 'train_size': 10_000_000, 'valid_size': 2_000,
    'tokenizer_train_size': 20_000, 'vocab_size': 8_000,
    'max_src_len': 256, 'max_tgt_len': 256,
    'd_model': 512, 'nhead': 8,
    'num_encoder_layers': 6, 'num_decoder_layers': 4,
    'dim_feedforward': 1088, 'dropout': 0.1,
    'batch_size': 16, 'global_batch_size': 32, 'world_size': 2,
    'epochs': 1, 'learning_rate': 3e-4, 'weight_decay': 0.01,
    'grad_clip': 1.0, 'log_every': 100, 'valid_every': 2000,
    'scheduler': 'cosine', 'warmup_ratio': 0.05,
    'norm_first': True, 'norm_eps': 1e-5,
    'ffn_type': 'mixed_dense_moe', 'expert_ffn_type': 'swiglu',
    'max_relative_distance': 128,
    'encoder_moe_layers': [1, 3], 'decoder_moe_layers': [1, 3],
    'moe_location': 'encoder_decoder', 'num_experts': 4,
    'moe_top_k': 2, 'moe_aux_coef': 0.01,
    'compile': True, 'use_amp': True,
    'checkpoint_every_minutes': 30, 'session_training_hours': 10.0,
    'purpose': 'Train a 59.78M-parameter MoE encoder-decoder from scratch on 10M examples using two-GPU DDP, a global batch of 32 and cosine learning-rate decay.',
}

resume_config = None
if resume_training:
    revision = api.model_info(repo_id).sha
    snapshot = snapshot_download(
        repo_id=repo_id, revision=revision, token=hf_token,
        allow_patterns=[f'{checkpoint_prefix}/{name}' for name in ['last.pt', 'best.pt', 'tokenizer.json']],
    )
    resume_folder = Path(snapshot) / checkpoint_prefix
    checkpoint = torch.load(resume_folder / 'last.pt', map_location='cpu', weights_only=True)
    resume_config = dict(checkpoint['config'])
    if checkpoint.get('world_size') != 2 or resume_config['d_model'] != 512:
        raise ValueError('This checkpoint does not belong to the two-GPU 60M experiment')
    config = dict(resume_config)
    for name in ['last.pt', 'best.pt', 'tokenizer.json']:
        source = resume_folder / name
        if source.exists():
            shutil.copy2(source, output_dir / name)
    tokenizer = Tokenizer.from_file(str(output_dir / 'tokenizer.json'))
    del checkpoint
else:
    if (output_dir / 'last.pt').exists() or (output_dir / 'best.pt').exists():
        raise RuntimeError('Existing checkpoints found. Set resume_training=True or choose a new workspace.')

In [ ]:
dataset_id = 'SlavaYus/tinysms_qa_12m'
dataset_revision = config.get('dataset_revision') or api.dataset_info(dataset_id).sha
config['dataset_revision'] = dataset_revision

def source_stream():
    return load_dataset(dataset_id, split='train', revision=dataset_revision, streaming=True)

def get_source_size():
    identity = {'dataset': dataset_id, 'revision': dataset_revision, 'split': 'train'}
    saved_fingerprint = config.get('data_fingerprint', {})
    if (
        saved_fingerprint.get('dataset') == dataset_id
        and saved_fingerprint.get('revision') == dataset_revision
        and saved_fingerprint.get('source_size', 0) > 0
    ):
        return int(saved_fingerprint['source_size'])

    count_path = workspace / 'source_row_count.json'
    if count_path.exists():
        saved_count = json.loads(count_path.read_text())
        if saved_count.get('identity') == identity and saved_count.get('rows', 0) > 0:
            return int(saved_count['rows'])

    stream = source_stream()
    splits = getattr(stream.info, 'splits', None)
    split_info = splits.get('train') if splits else None
    rows = getattr(split_info, 'num_examples', None)
    if not rows:
        print('Train row count is missing from metadata; counting the stream without saving raw data.')
        rows = sum(1 for _ in tqdm(stream, desc='Count train records', mininterval=30))
    if rows <= 0:
        raise RuntimeError('The training dataset is empty')

    temporary_path = count_path.with_suffix('.json.tmp')
    temporary_path.write_text(json.dumps({'identity': identity, 'rows': int(rows)}))
    os.replace(temporary_path, count_path)
    return int(rows)

source_size = get_source_size()
print('Source train records:', source_size)

n_test = math.ceil(source_size * 0.10)
assert source_size - n_test >= config['train_size']
assert n_test >= config['valid_size']
assert config['train_size'] % config['world_size'] == 0

permutation = np.random.default_rng(config['seed']).permutation(source_size)
train_indices = permutation[n_test:n_test + config['train_size']].copy()
valid_indices = permutation[:config['valid_size']].copy()
del permutation

if not resume_training:
    tokenizer_indices = train_indices[:min(config['tokenizer_train_size'], len(train_indices))]
    needed = set(tokenizer_indices.tolist())
    records = {}
    for row_id, row in enumerate(tqdm(source_stream(), total=source_size, desc='Read tokenizer sample', mininterval=30)):
        if row_id in needed:
            records[row_id] = (row['question'], row['code'])
            if len(records) == len(needed):
                break
    assert len(records) == len(needed)
    tokenizer = Tokenizer(models.BPE(unk_token='[UNK]'))
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=config['vocab_size'], min_frequency=2,
        special_tokens=['[PAD]', '[BOS]', '[EOS]', '[UNK]'],
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    )
    def tokenizer_iterator():
        for offset in range(0, len(tokenizer_indices), 1000):
            batch = [records[int(i)] for i in tokenizer_indices[offset:offset + 1000]]
            yield [r[0] for r in batch] + [r[1] for r in batch]
    tokenizer.train_from_iterator(tokenizer_iterator(), trainer=trainer)
    config['vocab_size'] = tokenizer.get_vocab_size()
    tokenizer.save(str(output_dir / 'tokenizer.json'))
    del records, needed

PAD_ID = tokenizer.token_to_id('[PAD]')
BOS_ID = tokenizer.token_to_id('[BOS]')
EOS_ID = tokenizer.token_to_id('[EOS]')
assert all(value is not None for value in (PAD_ID, BOS_ID, EOS_ID))
assert config['vocab_size'] == tokenizer.get_vocab_size()
assert config['vocab_size'] <= 65536

fingerprint = {
    'format': 'streamed_uint16_arrow_v1',
    'dataset': dataset_id, 'revision': dataset_revision,
    'source_size': source_size, 'seed': config['seed'],
    'train_size': config['train_size'], 'valid_size': config['valid_size'],
    'tokenizer_sha256': hashlib.sha256((output_dir / 'tokenizer.json').read_bytes()).hexdigest(),
    'max_src_len': config['max_src_len'], 'max_tgt_len': config['max_tgt_len'],
}
if resume_config is not None and resume_config.get('data_fingerprint') != fingerprint:
    raise ValueError('Checkpoint uses a different prepared-data format or dataset')
config['data_fingerprint'] = fingerprint

data_dir = workspace / 'data_compact'
data_dir.mkdir(parents=True, exist_ok=True)
manifest = data_dir / 'manifest.json'
reserve_bytes = 3 * 1024**3

if manifest.exists():
    if json.loads(manifest.read_text()) != fingerprint:
        raise ValueError('Prepared data differs; choose a new workspace')
else:
    if shutil.disk_usage(workspace).free < reserve_bytes + 1024**3:
        raise RuntimeError('Not enough free disk space. Start a clean Kaggle session.')
    membership = np.zeros(source_size, dtype=np.uint8)
    membership[train_indices] = 1
    membership[valid_indices] = 2
    schema = pa.schema([
        ('src_ids', pa.list_(pa.uint16())),
        ('tgt_ids', pa.list_(pa.uint16())),
    ])
    buffers = {1: [], 2: []}
    counts = {1: 0, 2: 0}
    def flush_buffer(kind, writer):
        rows = buffers[kind]
        if not rows:
            return
        if shutil.disk_usage(workspace).free < reserve_bytes:
            raise RuntimeError('Disk reserve reached. Free space before retrying preparation.')
        questions = tokenizer.encode_batch([r['question'] for r in rows])
        codes = tokenizer.encode_batch([r['code'] for r in rows])
        batch = pa.record_batch([
            pa.array([q.ids[:config['max_src_len'] - 1] + [EOS_ID] for q in questions], type=schema.field('src_ids').type),
            pa.array([[BOS_ID] + c.ids[:config['max_tgt_len'] - 2] + [EOS_ID] for c in codes], type=schema.field('tgt_ids').type),
        ], schema=schema)
        writer.write_batch(batch)
        counts[kind] += len(rows)
        rows.clear()
    with pa.OSFile(str(data_dir / 'train.arrow.tmp'), 'wb') as train_sink, pa.OSFile(str(data_dir / 'valid.arrow.tmp'), 'wb') as valid_sink:
        with pa.ipc.new_stream(train_sink, schema) as train_writer, pa.ipc.new_stream(valid_sink, schema) as valid_writer:
            writers = {1: train_writer, 2: valid_writer}
            for row_id, row in enumerate(tqdm(source_stream(), total=source_size, desc='Tokenize stream', mininterval=30)):
                if row_id >= source_size:
                    raise RuntimeError('Dataset row count changed')
                kind = int(membership[row_id])
                if kind:
                    buffers[kind].append(row)
                    if len(buffers[kind]) >= 1000:
                        flush_buffer(kind, writers[kind])
            for kind in (1, 2):
                flush_buffer(kind, writers[kind])
    assert counts[1] == config['train_size'] and counts[2] == config['valid_size']
    for name, indices in [('train', train_indices), ('valid', valid_indices)]:
        os.replace(data_dir / f'{name}.arrow.tmp', data_dir / f'{name}.arrow')
        physical_positions = np.searchsorted(np.sort(indices), indices)
        encoded = Dataset.from_file(str(data_dir / f'{name}.arrow'))
        reordered = encoded.select(
            physical_positions,
            indices_cache_file_name=str(data_dir / f'{name}_indices.arrow'),
        )
        assert len(reordered) == len(indices)
        del encoded, reordered, physical_positions
    manifest.write_text(json.dumps(fingerprint))
    del membership

del train_indices, valid_indices
config['global_batch_size'] = config['batch_size'] * config['world_size']
config['num_training_steps'] = math.ceil(config['train_size'] / config['global_batch_size']) * config['epochs']
config['num_warmup_steps'] = int(config['num_training_steps'] * config['warmup_ratio'])
settings = {
    'config': config, 'pad_id': PAD_ID, 'repo_id': repo_id,
    'checkpoint_prefix': checkpoint_prefix, 'experiment_name': experiment_name,
    'resume_training': resume_training,
}
(workspace / 'settings.json').write_text(json.dumps(settings, indent=2))
print('Train examples:', config['train_size'])
print('Global batch:', config['global_batch_size'])
print('Optimizer steps:', config['num_training_steps'])
print('Free disk GiB:', round(shutil.disk_usage(workspace).free / 1024**3, 2))
print('HF checkpoints:', f'{repo_id}/{checkpoint_prefix}')

In [ ]:
%%writefile /kaggle/working/moe60m_ddp/train.py
import os
import json
import math
import random
import time
from datetime import timedelta
from pathlib import Path
from functools import partial
from itertools import islice
from contextlib import nullcontext

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, DistributedSampler, Sampler
from datasets import Dataset
from transformers import get_cosine_schedule_with_warmup
from huggingface_hub import HfApi
from tqdm.auto import tqdm
import wandb
import deepspeed
from deepspeed.moe.layer import MoE


class BasicFeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.linear1 = nn.Linear(
            config['d_model'],
            config['dim_feedforward'],
        )
        self.linear2 = nn.Linear(
            config['dim_feedforward'],
            config['d_model'],
        )
        self.dropout = nn.Dropout(config['dropout'])
    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

class SwiGLUFeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()

        d_model = config["d_model"]
        hidden_dim = config["dim_feedforward"]

        self.gate_proj = nn.Linear(d_model, hidden_dim)
        self.up_proj = nn.Linear(d_model, hidden_dim)
        self.down_proj = nn.Linear(hidden_dim, d_model)
        self.dropout = nn.Dropout(config["dropout"])

    def forward(self, x):
        hidden = F.silu(self.gate_proj(x)) * self.up_proj(x)
        return self.down_proj(self.dropout(hidden))

class DenseFeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ffn = SwiGLUFeedForward(config)

    def forward(self, x, padding_mask=None):
        output = self.ffn(x)
        aux_loss = output.new_zeros((), dtype=torch.float32)
        return output, aux_loss, None

class MoEFeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.num_experts = config["num_experts"]

        self.moe = MoE(
            hidden_size=config["d_model"],
            expert=SwiGLUFeedForward(config),
            num_experts=self.num_experts,
            ep_size=1,
            k=config["moe_top_k"],
            drop_tokens=False,
            use_residual=False,
            noisy_gate_policy=None,
            top2_2nd_expert_sampling=False,
            use_tutel=False,
        )

        self.moe.set_deepspeed_parallelism()

    def forward(self, x, padding_mask=None):
        flat = x.reshape(-1, x.size(-1))

        valid = (
            torch.ones(
                flat.size(0),
                dtype=torch.bool,
                device=x.device,
            )
            if padding_mask is None
            else ~padding_mask.reshape(-1)
        )

        indices = valid.nonzero(as_tuple=True)[0]
        tokens = flat[indices]

        if tokens.size(0) == 0:
            return (
                torch.zeros_like(x),
                x.new_zeros((), dtype=torch.float32),
                x.new_zeros(self.num_experts, dtype=torch.float32),
            )

        output, aux_loss, counts = self.moe(tokens)

        result = torch.zeros_like(flat).index_copy(
            0,
            indices,
            output.to(flat.dtype),
        )

        counts = counts.detach().to(
            device=x.device,
            dtype=torch.float32,
        )
        load = counts / counts.sum().clamp_min(1)

        return result.view_as(x), aux_loss, load

class RelativePositionBias(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.max_distance = config["max_relative_distance"]
        self.num_heads = config["nhead"]

        self.embedding = nn.Embedding(
            2 * self.max_distance + 1,
            self.num_heads,
        )

    def forward(self, x, padding_mask=None, causal_mask=None):
        batch_size, length, _ = x.shape
        positions = torch.arange(length, device=x.device)

        distances = positions[None, :] - positions[:, None]
        indices = (
            distances.clamp(-self.max_distance, self.max_distance)
            + self.max_distance
        )

        bias = self.embedding(indices).permute(2, 0, 1)
        bias = bias.to(dtype=x.dtype)
        bias = bias.unsqueeze(0).expand(
            batch_size, -1, -1, -1
        )

        if padding_mask is not None:
            bias = bias.masked_fill(
                padding_mask[:, None, None, :],
                float("-inf"),
            )

        if causal_mask is not None:
            bias = bias.masked_fill(
                causal_mask[None, None, :, :],
                float("-inf"),
            )

        return bias.reshape(
            batch_size * self.num_heads,
            length,
            length,
        )

class EncoderLayer(nn.Module):
    def __init__(self, config, use_moe=False):
        super().__init__()

        self.position_bias = RelativePositionBias(config)
        self.self_attn = nn.MultiheadAttention(
            embed_dim=config['d_model'],
            num_heads=config['nhead'],
            dropout=config['dropout'],
            batch_first=True,
        )

        self.use_moe = use_moe
        self.ffn = (
            MoEFeedForward(config)
            if use_moe
            else DenseFeedForward(config)
        )
        self.norm1 = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )
        self.norm2 = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )
        self.dropout1 = nn.Dropout(config['dropout'])
        self.dropout2 = nn.Dropout(config['dropout'])

    def attention(self, x, padding_mask):
        attention_bias = self.position_bias(
            x,
            padding_mask=padding_mask,
        )

        output, _ = self.self_attn(
            query=x,
            key=x,
            value=x,
            attn_mask=attention_bias,
            need_weights=False,
        )
        return output

    def forward(self, x, padding_mask=None):
        x = x + self.dropout1(
            self.attention(self.norm1(x), padding_mask)
        )

        ffn_output, aux_loss, expert_load = self.ffn(
            self.norm2(x),
            padding_mask=padding_mask,
        )

        x = x + self.dropout2(ffn_output)

        return x, aux_loss, expert_load

class DecoderLayer(nn.Module):
    def __init__(self, config, use_moe=False):
        super().__init__()

        self.position_bias = RelativePositionBias(config)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=config['d_model'],
            num_heads=config['nhead'],
            dropout=config['dropout'],
            batch_first=True,
        )

        self.self_attn = nn.MultiheadAttention(
            embed_dim=config['d_model'],
            num_heads=config['nhead'],
            dropout=config['dropout'],
            batch_first=True,
        )

        self.use_moe = use_moe
        self.ffn = (
            MoEFeedForward(config)
            if use_moe
            else DenseFeedForward(config)
        )

        self.norm1 = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )
        self.norm2 = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )
        self.norm3 = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )

        self.dropout1 = nn.Dropout(config["dropout"])
        self.dropout2 = nn.Dropout(config["dropout"])
        self.dropout3 = nn.Dropout(config["dropout"])

    def self_attention(self, x, causal_mask, padding_mask):
        attention_bias = self.position_bias(
            x,
            padding_mask=padding_mask,
            causal_mask=causal_mask,
        )

        output, _ = self.self_attn(
            query=x,
            key=x,
            value=x,
            attn_mask=attention_bias,
            need_weights=False,
        )
        return output

    def cross_attention(self, x, memory, memory_padding_mask):
        output, _ = self.cross_attn(
            query=x,
            key=memory,
            value=memory,
            key_padding_mask=memory_padding_mask,
            need_weights=False,
        )
        return output

    def forward(
        self,
        x,
        memory,
        causal_mask,
        padding_mask=None,
        memory_padding_mask=None,
    ):
        x = x + self.dropout1(
            self.self_attention(self.norm1(x), causal_mask, padding_mask)
        )
        x = x + self.dropout2(
            self.cross_attention(self.norm2(x), memory, memory_padding_mask)
        )

        ffn_output, aux_loss, expert_load = self.ffn(
            self.norm3(x),
            padding_mask=padding_mask,
        )
        x = x + self.dropout3(ffn_output)

        return x, aux_loss, expert_load

class CodeTransformer(nn.Module):
    def __init__(self, config, pad_id):
        super().__init__()

        self.pad_id = pad_id
        self.d_model = config["d_model"]

        self.token_embedding = nn.Embedding(
            config["vocab_size"],
            self.d_model,
            padding_idx=pad_id,
        )
        self.num_experts = config["num_experts"]
        self.max_src_len = config["max_src_len"]
        self.max_tgt_len = config["max_tgt_len"]

        self.embedding_dropout = nn.Dropout(config["dropout"])

        self.encoder_layers = nn.ModuleList([
            EncoderLayer(
                config,
                use_moe=i in config["encoder_moe_layers"],
            )
            for i in range(config["num_encoder_layers"])
        ])

        self.decoder_layers = nn.ModuleList([
            DecoderLayer(
                config,
                use_moe=i in config["decoder_moe_layers"],
            )
            for i in range(config["num_decoder_layers"])
        ])

        self.encoder_norm = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )
        self.decoder_norm = nn.RMSNorm(
            config['d_model'],
            eps=config['norm_eps']
        )

        self.output = nn.Linear(
            self.d_model,
            config["vocab_size"],
        )

        self.reset_parameters()

    def reset_parameters(self):
        for layers in (self.encoder_layers, self.decoder_layers):
            for parameter in layers.parameters():
                if parameter.dim() > 1:
                    nn.init.xavier_uniform_(parameter)

        nn.init.normal_(self.token_embedding.weight, std=0.02)

        for module in self.modules():
            if isinstance(module, RelativePositionBias):
                nn.init.zeros_(module.embedding.weight)

        with torch.no_grad():
            self.token_embedding.weight[self.pad_id].zero_()

    def collect_aux(self, x, aux_losses, expert_loads):
        if not aux_losses:
            return (
                x.new_zeros((), dtype=torch.float32),
                x.new_zeros((0, self.num_experts), dtype=torch.float32),
            )

        return (
            torch.stack(aux_losses).mean(),
            torch.stack(expert_loads),
        )

    def embed(self, ids):
        x = self.token_embedding(ids) * math.sqrt(self.d_model)
        return self.embedding_dropout(x)

    def encode(self, src, return_aux=False):
        padding_mask = src.eq(self.pad_id)
        memory = self.embed(src)

        aux_losses = []
        expert_loads = []

        for layer in self.encoder_layers:
            memory, aux_loss, expert_load = layer(
                memory,
                padding_mask=padding_mask,
            )

            if return_aux and layer.use_moe:
                aux_losses.append(aux_loss)
                expert_loads.append(expert_load)

        memory = self.encoder_norm(memory)

        if return_aux:
            aux_loss, loads = self.collect_aux(
                memory, aux_losses, expert_loads
            )
            return memory, aux_loss, loads

        return memory


    def decode(
        self,
        tgt_input,
        memory,
        memory_padding_mask=None,
        return_aux=False,
    ):
        length = tgt_input.size(1)

        causal_mask = torch.triu(
            torch.ones(
                length,
                length,
                dtype=torch.bool,
                device=tgt_input.device,
            ),
            diagonal=1,
        )

        padding_mask = tgt_input.eq(self.pad_id)
        x = self.embed(tgt_input)

        aux_losses = []
        expert_loads = []

        for layer in self.decoder_layers:
            x, aux_loss, expert_load = layer(
                x,
                memory,
                causal_mask=causal_mask,
                padding_mask=padding_mask,
                memory_padding_mask=memory_padding_mask,
            )

            if return_aux and layer.use_moe:
                aux_losses.append(aux_loss)
                expert_loads.append(expert_load)

        logits = self.output(self.decoder_norm(x))

        if return_aux:
            aux_loss, loads = self.collect_aux(
                x, aux_losses, expert_loads
            )
            return logits, aux_loss, loads

        return logits


    def forward(self, src, tgt_input, return_aux=False):
        if not return_aux:
            memory = self.encode(src)
            return self.decode(
                tgt_input,
                memory,
                memory_padding_mask=src.eq(self.pad_id),
            )

        memory, encoder_aux, encoder_loads = self.encode(
            src,
            return_aux=True,
        )

        logits, decoder_aux, decoder_loads = self.decode(
            tgt_input,
            memory,
            memory_padding_mask=src.eq(self.pad_id),
            return_aux=True,
        )

        n_encoder = encoder_loads.size(0)
        n_decoder = decoder_loads.size(0)

        aux_loss = (
            encoder_aux * n_encoder + decoder_aux * n_decoder
        ) / max(n_encoder + n_decoder, 1)

        expert_loads = torch.cat(
            [encoder_loads, decoder_loads],
            dim=0,
        )

        return logits, aux_loss, expert_loads


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class ResumeBatchSampler(Sampler):
    def __init__(self, sampler, batch_size):
        self.sampler = sampler
        self.batch_size = batch_size
        self.start_batch = 0

    def __iter__(self):
        indices = islice(iter(self.sampler), self.start_batch * self.batch_size, None)
        while True:
            batch = list(islice(indices, self.batch_size))
            if not batch:
                return
            yield batch

    def __len__(self):
        total = math.ceil(len(self.sampler) / self.batch_size)
        return max(0, total - self.start_batch)


def collate(examples, pad_id):
    src = pad_sequence(
        [torch.tensor(x['src_ids'], dtype=torch.long) for x in examples],
        batch_first=True, padding_value=pad_id,
    )
    tgt = pad_sequence(
        [torch.tensor(x['tgt_ids'], dtype=torch.long) for x in examples],
        batch_first=True, padding_value=pad_id,
    )
    return src, tgt


def amp_context(device, enabled):
    return torch.autocast('cuda', dtype=torch.float16) if enabled else nullcontext()


def capture_rng(loader_generator, device):
    state = np.random.get_state()
    return {
        'python': random.getstate(),
        'numpy': [state[0], state[1].tolist(), state[2], state[3], state[4]],
        'torch': torch.get_rng_state(),
        'cuda': torch.cuda.get_rng_state(device) if device.type == 'cuda' else None,
        'loader': loader_generator.get_state(),
    }


def restore_rng(state, loader_generator, device):
    random.setstate(state['python'])
    ns = state['numpy']
    np.random.set_state((ns[0], np.asarray(ns[1], dtype=np.uint32), ns[2], ns[3], ns[4]))
    torch.set_rng_state(state['torch'].cpu())
    if state['cuda'] is not None:
        torch.cuda.set_rng_state(state['cuda'].cpu(), device)
    loader_generator.set_state(state['loader'].cpu())


def atomic_save(payload, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(payload, temporary)
    os.replace(temporary, path)


@torch.inference_mode()
def evaluate(model, loader, device, pad_id, use_amp):
    model.eval()
    totals = torch.zeros(3, dtype=torch.float64, device=device)
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        labels = tgt[:, 1:]
        with amp_context(device, use_amp):
            logits = model(src, tgt[:, :-1])
            loss_sum = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)), labels.reshape(-1),
                ignore_index=pad_id, reduction='sum',
            )
        mask = labels.ne(pad_id)
        totals[0] += loss_sum.double()
        totals[1] += ((logits.argmax(-1) == labels) & mask).sum()
        totals[2] += mask.sum()
    dist.all_reduce(totals)
    if not torch.isfinite(totals).all().item() or totals[2].item() == 0:
        raise RuntimeError('Invalid validation metrics')
    return {
        'val/loss': (totals[0] / totals[2]).item(),
        'val/token_accuracy': (totals[1] / totals[2]).item(),
    }


def train_ddp(model, base_model, train_loader, valid_loader, batch_sampler,
              optimizer, scheduler, scaler, config, device, pad_id, output_dir,
              run=None, upload=None, checkpoint=None):
    rank, world_size = dist.get_rank(), dist.get_world_size()
    output_dir = Path(output_dir)
    total_batches = math.ceil(len(batch_sampler.sampler) / batch_sampler.batch_size)
    global_step, start_epoch, start_batch = 0, 1, 0
    best_val_loss = float('inf')
    history = []
    if checkpoint is not None:
        global_step = checkpoint['global_step']
        start_epoch = checkpoint['epoch']
        start_batch = checkpoint['batch_idx']
        best_val_loss = checkpoint['best_val_loss']
        history = list(checkpoint['history'])
    last_save = time.monotonic()
    started = time.monotonic()
    stop_budget = config.get('session_training_hours', 0) * 3600
    if start_epoch > config['epochs']:
        return 'complete'

    def save(epoch, batch_idx, epoch_totals, status):
        nonlocal last_save
        state = capture_rng(train_loader.generator, device)
        states = [None] * world_size
        dist.all_gather_object(states, state)
        if rank == 0:
            payload = {
                'model_state_dict': base_model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'scaler_state_dict': scaler.state_dict(),
                'config': dict(config), 'pad_id': pad_id,
                'epoch': epoch, 'batch_idx': batch_idx,
                'global_step': global_step, 'best_val_loss': best_val_loss,
                'epoch_totals': epoch_totals, 'rng_states': states,
                'world_size': world_size, 'history': history,
                'wandb_run_id': run.id if run is not None else None,
                'status': status,
            }
            atomic_save(payload, output_dir / 'last.pt')
            if upload is not None:
                upload(output_dir)
            print(f'Checkpoint: epoch={epoch}, batch={batch_idx}, status={status}', flush=True)
        dist.barrier()
        last_save = time.monotonic()

    def validate(epoch):
        nonlocal best_val_loss
        metrics = evaluate(base_model, valid_loader, device, pad_id, config['use_amp'])
        if metrics['val/loss'] < best_val_loss:
            best_val_loss = metrics['val/loss']
            if rank == 0:
                atomic_save({
                    'model_state_dict': base_model.state_dict(),
                    'config': dict(config), 'pad_id': pad_id,
                    'epoch': epoch, 'global_step': global_step,
                    'val_loss': best_val_loss,
                }, output_dir / 'best.pt')
                if run is not None:
                    run.summary['best/val_loss'] = best_val_loss
        dist.barrier()
        model.train()
        return metrics

    for epoch in range(start_epoch, config['epochs'] + 1):
        batch_sampler.sampler.set_epoch(epoch)
        resuming = checkpoint is not None and epoch == start_epoch
        skip = start_batch if resuming else 0
        batch_sampler.start_batch = skip
        epoch_totals = torch.zeros(2, dtype=torch.float64, device=device)
        if resuming:
            epoch_totals.copy_(torch.tensor(checkpoint['epoch_totals'], dtype=torch.float64, device=device))
        iterator = iter(train_loader)
        if resuming:
            restore_rng(checkpoint['rng_states'][rank], train_loader.generator, device)
        model.train()
        window = torch.zeros(5, dtype=torch.float64, device=device)
        window_start = time.perf_counter()
        progress = tqdm(iterator, total=total_batches, initial=skip,
                        desc=f'Epoch {epoch}', disable=rank != 0, mininterval=30)
        for batch_idx, (src, tgt) in enumerate(progress, skip + 1):
            src, tgt = src.to(device, non_blocking=True), tgt.to(device, non_blocking=True)
            labels = tgt[:, 1:]
            optimizer.zero_grad(set_to_none=True)
            local_tokens = labels.ne(pad_id).sum()
            global_tokens = local_tokens.clone()
            dist.all_reduce(global_tokens)
            with amp_context(device, config['use_amp']):
                logits, aux_loss, expert_loads = model(src, tgt[:, :-1], return_aux=True)
                loss_sum = F.cross_entropy(
                    logits.reshape(-1, logits.size(-1)), labels.reshape(-1),
                    ignore_index=pad_id, reduction='sum',
                )
                loss = loss_sum * world_size / global_tokens.clamp_min(1)
                total_loss = loss + config['moe_aux_coef'] * aux_loss
            finite = torch.isfinite(total_loss).int()
            dist.all_reduce(finite, op=dist.ReduceOp.MIN)
            if not finite.item():
                raise RuntimeError('Non-finite training loss')
            scaler.scale(total_loss).backward()
            scaler.unscale_(optimizer)
            grad_norm = nn.utils.clip_grad_norm_(base_model.parameters(), config['grad_clip'])
            lr_used = optimizer.param_groups[0]['lr']
            scale_before = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() >= scale_before:
                scheduler.step()
                global_step += 1

            batch_totals = torch.stack([loss_sum.detach().double(), local_tokens.double()])
            dist.all_reduce(batch_totals)
            epoch_totals += batch_totals
            window[0] += loss_sum.detach().double()
            window[1] += local_tokens.double()
            window[2] += aux_loss.detach().double()
            window[3] += total_loss.detach().double()
            window[4] += 1
            if batch_idx % config['log_every'] == 0 or batch_idx == total_batches:
                metrics_sum = window.clone()
                dist.all_reduce(metrics_sum)
                loads = expert_loads.detach().float().clone()
                dist.all_reduce(loads)
                loads /= world_size
                if rank == 0:
                    elapsed = time.perf_counter() - window_start
                    metrics = {
                        'global_step': global_step, 'epoch': epoch,
                        'train/loss': (metrics_sum[0] / metrics_sum[1]).item(),
                        'train/moe_aux_loss': (metrics_sum[2] / metrics_sum[4]).item(),
                        'train/total_loss': (metrics_sum[3] / metrics_sum[4]).item(),
                        'train/lr': lr_used, 'train/grad_norm': grad_norm.item(),
                        'train/target_tokens_per_second': metrics_sum[1].item() / elapsed,
                    }
                    names = [('encoder', i) for i in config['encoder_moe_layers']]
                    names += [('decoder', i) for i in config['decoder_moe_layers']]
                    for (stack, layer), values in zip(names, loads.cpu().tolist()):
                        for expert, value in enumerate(values):
                            metrics[f'moe/{stack}_{layer}/expert_{expert}'] = value
                    if run is not None:
                        run.log(metrics)
                    progress.set_postfix(loss=metrics['train/loss'], lr=lr_used)
                window.zero_()
                window_start = time.perf_counter()
            if batch_idx % config['valid_every'] == 0 and batch_idx != total_batches:
                before = time.perf_counter()
                metrics = validate(epoch)
                if rank == 0 and run is not None:
                    run.log({**metrics, 'global_step': global_step, 'epoch': epoch})
                window_start += time.perf_counter() - before

            flags = torch.zeros(2, dtype=torch.int32, device=device)
            if rank == 0:
                flags[0] = time.monotonic() - last_save >= config['checkpoint_every_minutes'] * 60
                flags[1] = bool(stop_budget and time.monotonic() - started >= stop_budget)
            dist.broadcast(flags, src=0)
            should_save, should_stop = flags.cpu().tolist()
            if (should_save or should_stop) and batch_idx != total_batches:
                before = time.perf_counter()
                save(epoch, batch_idx, epoch_totals.cpu().tolist(), 'paused' if should_stop else 'running')
                window_start += time.perf_counter() - before
                if should_stop:
                    return 'paused'
        metrics = validate(epoch)
        metrics.update({'epoch': epoch, 'global_step': global_step,
                        'train/epoch_loss': (epoch_totals[0] / epoch_totals[1]).item()})
        history.append(metrics)
        if rank == 0 and run is not None:
            run.log(metrics)
        save(epoch + 1, 0, [0.0, 0.0], 'complete' if epoch == config['epochs'] else 'running')
        checkpoint = None
    return 'complete'


def main():
    rank = int(os.environ['RANK'])
    local_rank = int(os.environ['LOCAL_RANK'])
    world_size = int(os.environ['WORLD_SIZE'])
    if world_size != 2:
        raise ValueError('Launch with exactly two processes')
    torch.cuda.set_device(local_rank)
    device = torch.device('cuda', local_rank)
    deepspeed.init_distributed(
        dist_backend='nccl', auto_mpi_discovery=False,
        init_method='env://', timeout=timedelta(hours=2),
    )
    workspace = Path(os.environ['MOE60_WORKSPACE'])
    settings = json.loads((workspace / 'settings.json').read_text())
    config = settings['config']
    output_dir = workspace / 'output'
    checkpoint = None
    if settings['resume_training']:
        checkpoint = torch.load(output_dir / 'last.pt', map_location='cpu', weights_only=True)
        if checkpoint['world_size'] != world_size:
            raise ValueError('Resume requires the original world size')
        for key, value in checkpoint['config'].items():
            if key not in ('session_training_hours', 'checkpoint_every_minutes', 'compile'):
                if config.get(key) != value:
                    raise ValueError(f'Resume config mismatch: {key}')
    set_seed(config['seed'])
    base_model = CodeTransformer(config, settings['pad_id']).to(device)
    optimizer = torch.optim.AdamW(base_model.parameters(), lr=config['learning_rate'],
                                  weight_decay=config['weight_decay'])
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=config['num_warmup_steps'],
        num_training_steps=config['num_training_steps'],
    )
    scaler = torch.amp.GradScaler('cuda', enabled=config['use_amp'])
    if checkpoint is not None:
        base_model.load_state_dict(checkpoint['model_state_dict'], strict=True)
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
    if config['compile']:
        base_model.compile()
    model = DistributedDataParallel(
        base_model, device_ids=[local_rank], output_device=local_rank,
        broadcast_buffers=False, find_unused_parameters=True,
        gradient_as_bucket_view=True,
    )
    set_seed(config['seed'] + rank)
    train_data = Dataset.from_file(
        str(workspace / 'data_compact' / 'train.arrow'),
        indices_filename=str(workspace / 'data_compact' / 'train_indices.arrow'),
    )
    valid_data = Dataset.from_file(
        str(workspace / 'data_compact' / 'valid.arrow'),
        indices_filename=str(workspace / 'data_compact' / 'valid_indices.arrow'),
    )
    if len(train_data) != config['train_size'] or len(valid_data) != config['valid_size']:
        raise ValueError('Prepared dataset size mismatch')
    sampler = DistributedSampler(train_data, num_replicas=world_size, rank=rank,
                                 shuffle=True, seed=config['seed'], drop_last=False)
    batch_sampler = ResumeBatchSampler(sampler, config['batch_size'])
    loader_generator = torch.Generator().manual_seed(config['seed'] + rank)
    train_loader = DataLoader(
        train_data, batch_sampler=batch_sampler,
        collate_fn=partial(collate, pad_id=settings['pad_id']),
        num_workers=0, pin_memory=True, generator=loader_generator,
    )
    valid_loader = DataLoader(
        valid_data.select(range(rank, len(valid_data), world_size)),
        batch_size=config['batch_size'], shuffle=False,
        collate_fn=partial(collate, pad_id=settings['pad_id']),
        num_workers=0, pin_memory=True,
    )
    if len(train_loader) * config['epochs'] != config['num_training_steps']:
        raise ValueError('Incorrect scheduler length')
    run = None
    api = None
    if rank == 0:
        api = HfApi(token=os.environ['HF_TOKEN'])
        run = wandb.init(
            project='hw2-code-generation', name=settings['experiment_name'],
            group=settings['experiment_name'], config=config,
        )
        run.summary['model/parameters'] = sum(p.numel() for p in base_model.parameters())
        run.summary['torch_version'] = str(torch.__version__)
        run.summary['deepspeed_version'] = str(deepspeed.__version__)
        run.summary['world_size'] = world_size
        print('Parameters:', run.summary['model/parameters'], flush=True)
    dist.barrier()
    def upload(folder):
        api.upload_folder(
            repo_id=settings['repo_id'], repo_type='model', folder_path=str(folder),
            path_in_repo=settings['checkpoint_prefix'],
            allow_patterns=['last.pt', 'best.pt', 'tokenizer.json'],
            commit_message='Save 60M DDP training checkpoint',
        )
    try:
        status = train_ddp(
            model, base_model, train_loader, valid_loader, batch_sampler,
            optimizer, scheduler, scaler, config, device, settings['pad_id'],
            output_dir, run=run, upload=upload if rank == 0 else None, checkpoint=checkpoint,
        )
        if rank == 0:
            (workspace / 'status.json').write_text(json.dumps({'status': status}))
            run.finish()
    finally:
        dist.destroy_process_group()


if __name__ == '__main__':
    main()

In [ ]:
child_env = dict(os.environ)
child_env.update({
    'HF_TOKEN': hf_token,
    'WANDB_API_KEY': wandb_token,
    'MOE60_WORKSPACE': str(workspace),
    'TOKENIZERS_PARALLELISM': 'false',
    'OMP_NUM_THREADS': '1',
    'PYTHONUNBUFFERED': '1',
})
for key in ['RANK', 'WORLD_SIZE', 'LOCAL_RANK', 'LOCAL_WORLD_SIZE', 'MASTER_ADDR', 'MASTER_PORT']:
    child_env.pop(key, None)

command = [
    sys.executable, '-m', 'torch.distributed.run',
    '--standalone', '--nnodes=1', '--nproc-per-node=2',
    str(workspace / 'train.py'),
]
log_path = workspace / 'train.log'
process = subprocess.Popen(
    command, env=child_env, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True, bufsize=1,
)
try:
    with log_path.open('a', encoding='utf-8') as log:
        for line in process.stdout:
            print(line, end='')
            log.write(line)
    return_code = process.wait()
except KeyboardInterrupt:
    process.terminate()
    process.wait()
    raise
finally:
    child_env.pop('HF_TOKEN', None)
    child_env.pop('WANDB_API_KEY', None)
if return_code:
    raise RuntimeError(f'Distributed training failed with code {return_code}; see {log_path}')
print(json.loads((workspace / 'status.json').read_text()))